<a href="https://colab.research.google.com/github/ingkapat/t/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thai Scam Detection Dataset Generator

Synthetic data generator for binary classification (scam vs not-scam).
Uses GPT-4o mini via OpenAI API.

## Schema
```
{
  "label": 0 | 1,
  "turns": [{"speaker": "caller"|"user", "text": str}, ...]
}
```

## Categories
- **Scam (label=1)**: 13 subtypes covering callcenter, investment, romance, phishing, etc.
- **Not-scam (label=0)**: general chat, verification codes, official calls, hard negatives, unknown calls

## Variation Matrix
Each record is generated with controlled variation across:
- 5 victim awareness levels (naive, skeptical, aware, busy, elderly)
- 3 language styles (formal, casual, mixed)
- 13+ subtype patterns

Total combinations cycled deterministically to ensure coverage.

## Target size: ~500 records

In [1]:
# Cell 1: Install dependencies
!pip install openai tqdm -q

In [2]:
# Cell 2: Configuration
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Quantity per subtype. Total = sum of (subtype_count * per_subtype)
CONFIG = {
    # Scam categories (label=1)
    'per_scam_short':     20,  # 6 subtypes x 20 = 120
    'per_scam_long':      18,  # 7 subtypes x 18 = 126

    # Not-scam categories (label=0)
    'per_general_chat':   16,  # 4 subtypes x 16 = 64
    'per_verification':   14,  # 3 subtypes x 14 = 42
    'per_official':       14,  # 4 subtypes x 14 = 56
    'per_hard_negative':  16,  # 4 subtypes x 16 = 64
    'per_unknown':         8,  # 4 subtypes x 8  = 32

    # Conversation length
    'min_turns': 1,
    'max_turns': 6,

    # API parameters
    'model':       'gpt-4o-mini',
    'temperature': 1.0,
    'max_retries': 3,
    'request_delay_sec': 0.15,

    # Output
    'output_dir':  '/content/dataset_output',
    'output_file': 'dataset.jsonl',
}

# Subtype definitions for scam category
SCAM_SHORT_SUBTYPES = [
    'reward', 'phishing', 'otp_hijack', 'sextortion', 'loan_scam', 'sms_alert',
]
SCAM_LONG_SUBTYPES = [
    'callcenter', 'investment', 'romance', 'tech_support',
    'job_scam', 'money_mule', 'impersonation',
]

# Subtypes for not-scam categories
GENERAL_CHAT_SUBTYPES   = ['friend_hangout', 'family_check', 'couple_talk', 'colleague']
VERIFICATION_SUBTYPES   = ['otp_thai', 'otp_english', 'reset_code']
OFFICIAL_SUBTYPES       = ['bank_real', 'gov_notice', 'hospital', 'company_hr']
HARD_NEGATIVE_SUBTYPES  = ['real_otp', 'real_job', 'friend_borrow', 'real_parcel']
UNKNOWN_SUBTYPES        = ['silent_call', 'wrong_number', 'connection_issue', 'incomplete_speech']

# Calculate target dataset size
n_scam = (CONFIG['per_scam_short'] * len(SCAM_SHORT_SUBTYPES) +
          CONFIG['per_scam_long']  * len(SCAM_LONG_SUBTYPES))
n_chat = CONFIG['per_general_chat'] * len(GENERAL_CHAT_SUBTYPES)
n_ver  = CONFIG['per_verification'] * len(VERIFICATION_SUBTYPES)
n_off  = CONFIG['per_official']     * len(OFFICIAL_SUBTYPES)
n_hard = CONFIG['per_hard_negative']* len(HARD_NEGATIVE_SUBTYPES)
n_unk  = CONFIG['per_unknown']      * len(UNKNOWN_SUBTYPES)
n_total = n_scam + n_chat + n_ver + n_off + n_hard + n_unk

print('Target dataset composition')
print(f'  Scam (label=1)         : {n_scam}')
print(f'  General chat (label=0) : {n_chat}')
print(f'  Verification (label=0) : {n_ver}')
print(f'  Official (label=0)     : {n_off}')
print(f'  Hard negative (label=0): {n_hard}')
print(f'  Unknown (label=0)      : {n_unk}')
print(f'  Total                  : {n_total}')
print(f'  Scam ratio             : {n_scam/n_total*100:.1f}%')

Target dataset composition
  Scam (label=1)         : 246
  General chat (label=0) : 64
  Verification (label=0) : 42
  Official (label=0)     : 56
  Hard negative (label=0): 64
  Unknown (label=0)      : 32
  Total                  : 504
  Scam ratio             : 48.8%


In [3]:
# Cell 3: Pattern definitions for content generation

# Scam patterns - short format (SMS/announcement style)
SCAM_SHORT_PATTERNS = {
    'reward':     'Notification claiming user won iPhone/cash/gold prize. '
                  'Requests contact via Line, link click, or ID card photo.',
    'phishing':   'SMS/notification about account locked or parcel delivery failure. '
                  'Pushes user to click bit.ly link or verify credentials.',
    'otp_hijack': 'Caller impersonates bank, requests 6-digit OTP code for verification. '
                  'Will use OTP to drain account.',
    'sextortion': 'Threatens to release private photos/videos unless payment is sent.',
    'loan_scam':  'Offers loan approval without credit check. '
                  'Requires upfront fee or ID document submission.',
    'sms_alert':  'Claims tax refund or social security benefit. '
                  'Requires dialing USSD code or submitting personal data.',
}

# Scam patterns - long format (multi-turn conversation)
SCAM_LONG_PATTERNS = {
    'callcenter':    'Call center scam impersonating police, DSI, or revenue department. '
                     'Claims account is linked to criminal case. Demands transfer to safe account. '
                     'Forbids hanging up or telling anyone.',
    'investment':   'Investment scam guaranteeing 30-50% monthly return with no risk. '
                     'Requests user to chat or click link to register and transfer funds urgently.',
    'romance':       'Romance scam with affectionate messages. '
                     'Claims to be foreign businessman, soldier, or doctor. '
                     'Eventually requests emergency loan transfer.',
    'tech_support':  'Impersonates Microsoft, Apple, or bank technical support. '
                     'Reports virus or hacked account. '
                     'Instructs user to install AnyDesk or TeamViewer for remote access.',
    'job_scam':      'Online job offer for liking videos or completing tasks. '
                     'Initially pays small amounts, then requires deposits to unlock withdrawals.',
    'money_mule':    'Offers commission for receiving and forwarding bank transfers. '
                     'Claims to be foreign company needing local Thai accounts.',
    'impersonation': 'Impersonates friend or relative claiming new phone number. '
                     'Creates emergency requiring urgent money transfer. '
                     'Refuses callbacks to old number.',
}

# Not-scam patterns
GENERAL_CHAT_PATTERNS = {
    'friend_hangout': 'Friend invites for shopping, food, or movies. Casual language with กู มึง แก เธอ.',
    'family_check':   'Family member checking on user, asking about meals or work. Warm tone.',
    'couple_talk':    'Partner calling to chat, set meeting, or express affection.',
    'colleague':      'Coworker discussing meetings, work tasks, or lunch plans.',
}

VERIFICATION_PATTERNS = {
    'otp_thai':    'Automated SMS/voice message in Thai delivering OTP code. '
                   'Includes warning not to share with anyone.',
    'otp_english': 'Automated message in English. Format: '
                   '"Your verification code is XXXXXX. Do not share."',
    'reset_code':  'Password reset confirmation code or 2FA code delivery.',
}

OFFICIAL_PATTERNS = {
    'bank_real':  'Bank notification about credit card statement, expiring card, or transaction confirmation. '
                  'Does not request OTP or sensitive information.',
    'gov_notice': 'Government office (revenue dept, social security, transport) announcing rights or appointments. '
                  'Does not request payment.',
    'hospital':   'Hospital calling for appointment scheduling, lab results, or rescheduling.',
    'company_hr': 'Real company HR scheduling job interview. Identifies company name and position clearly.',
}

HARD_NEGATIVE_PATTERNS = {
    'real_otp':      'Real bank requesting OTP for transaction confirmation. '
                     'Includes reference number. Does not ask for additional credentials.',
    'real_job':      'Real HR scheduling urgent interview because position closing soon. '
                     'Does not request payment or ID documents.',
    'friend_borrow': 'Close friend genuinely borrowing money. Uses informal pronouns กู/มึง. '
                     'Discusses other topics first. Provides personal context.',
    'real_parcel':   'Real delivery service notifying about package. Includes tracking number. '
                     'Does not send links or request payment.',
}

# Unknown patterns - calls with no clear content (silent, wrong number, technical issues)
UNKNOWN_PATTERNS = {
    'silent_call':       'Caller does not speak. May only have background noise or breathing. '
                          'User says ฮัลโหล or asks who is calling. No response.',
    'wrong_number':      'Caller asks for someone who is not at this number. '
                          'Brief exchange clarifying wrong number. Caller hangs up or apologizes.',
    'connection_issue':  'Phone connection has noise, echo, or breaking up. '
                          'Conversation cannot proceed. Both parties say cannot hear.',
    'incomplete_speech': 'Caller says incomplete sentences, mumbles, or speech is cut off. '
                          'Content is unclear and cannot be classified.',
}

# Variation dimensions
VICTIM_STYLES = {
    'naive':     'Trusting and easily persuaded. Brief positive responses like อือ, อ๋อ, โอเค, จริงเหรอ.',
    'skeptical': 'Mildly suspicious. Asks clarifying questions like แบบว่า, เอ๊ะ, รอก่อนนะ, แน่ใจเหรอ.',
    'aware':     'Recognizes the scam pattern. Cuts off with เฮ้ย หยุดก่อน, ไม่เอาแล้ว, จะวางสายนะ.',
    'busy':      'Busy or in a hurry. Short responses like รีบหน่อย, แป๊บนึง, กำลังขับรถ, มีอะไรเร็วๆ.',
    'elderly':   'Older person, slow to respond. Says อะไรนะ, พูดอีกทีได้ไหม, ฟังไม่ค่อยชัด.',
}

LANGUAGE_STYLES = {
    'formal': 'Standard Thai with polite particles ครับ/ค่ะ.',
    'casual': 'Colloquial Thai with particles อ่ะ, นะ, จ้า, เอ้า, อือ. Short sentences.',
    'mixed':  'Mixed Thai-English code-switching. Words like OK, confirm, check, sure.',
}

print('Patterns loaded')

Patterns loaded


In [4]:
# Cell 4: System prompt and prompt builders

SYSTEM_PROMPT = '''You are a Thai language conversation writer specializing in realistic phone dialogues.
Your task is to generate synthetic phone conversations for training a scam detection AI.

Critical requirement: Generated dialogue must sound like authentic Thai phone conversations,
not formal written language.

Guidelines:

1. Use realistic spoken Thai with appropriate sentence-final particles:
   ครับ, ค่ะ, นะ, น่ะ, อ่ะ, เว้ย, จ้า, ดิ
   อือ, อ๋อ, อา, แหม, เฮ้ย, อ้าว, โอ้โห
   แบบว่า, คือ, งั้น, เดี๋ยว

2. For close friends and family relationships, use informal pronouns:
   กู, มึง, เอ็ง, แก, เธอ, ไอ้..., อี่...

3. Keep utterances short and natural, not full sentences:
   Bad:  "ผมคิดว่าฟังดูน่าสนใจมากเลยครับ"
   Good: "อือ น่าสนใจอะ"

4. Scammers typically use overly formal language with words like กรุณา, โปรด, เรียน.

5. The user (call recipient) typically speaks less than the caller, especially in scam scenarios.

6. Some conversations may have only the caller speaking (broadcast/SMS style),
   with no user response. This is acceptable.

Reference examples:

Friend invitation:
  [caller: มึงว่างป่ะ ไปกินชาบูกัน]
  [user: ได้ๆ กี่โมงล่ะ]
  [caller: สัก 6 โมงเย็น]

OTP delivery (single-turn):
  [caller: รหัส OTP ของคุณคือ 123456 กรุณาอย่าเผยแพร่]

Scam SMS:
  [caller: คุณเป็นผู้โชคดีได้รับ iPhone 16 ฟรี 1 เครื่องค่ะ]
  [user: จริงหรอคะ]
  [caller: จริงค่ะ ติดต่อ Line @iDle23 เพื่อแจ้งที่อยู่]

Wrong number:
  [caller: ฮัลโหล สมศักดิ์ใช่ไหมครับ]
  [user: ไม่ใช่ครับ ผิดเบอร์]
  [caller: ขอโทษครับ]

Output format: JSON object with key "turns" only. No additional text.'''

TURNS_HINT = '{"turns": [{"speaker": "caller", "text": "..."}]}'

def build_prompt(category, subtype, victim_style, lang_style, min_t, max_t, extra=''):
    """
    Build user prompt for a single conversation generation request.

    Args:
        category:     Top-level category description.
        subtype:      Specific subtype description.
        victim_style: Victim awareness profile (or None for non-scam types).
        lang_style:   Language style descriptor.
        min_t, max_t: Min/max number of turns.
        extra:        Additional instructions specific to subtype.

    Returns:
        Formatted prompt string.
    """
    victim_section = f'Recipient profile: {victim_style}\n' if victim_style else ''

    return f'''Generate one phone conversation:

Category: {category}
Subtype: {subtype}
{victim_section}Language style: {lang_style}
Target length: {min_t}-{max_t} turns
{extra}

Language must sound natural for Thai phone conversation, not written prose.

Output JSON: {TURNS_HINT}'''

print('Prompt builder ready')

Prompt builder ready


In [5]:
# Cell 5: API client and helpers
from openai import OpenAI
import json, time

client = OpenAI()

def call_gpt(prompt, max_tokens=1000):
    """Call OpenAI Chat Completion API with JSON mode enabled."""
    response = client.chat.completions.create(
        model=CONFIG['model'],
        temperature=CONFIG['temperature'],
        max_tokens=max_tokens,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ],
    )
    return response.choices[0].message.content.strip()

def parse_turns(text):
    """
    Parse and validate JSON response from GPT.

    Returns:
        List of turn dicts with 'speaker' and 'text' keys.

    Raises:
        AssertionError if structure is invalid.
    """
    data = json.loads(text)
    turns = data['turns'] if isinstance(data, dict) and 'turns' in data else data
    assert isinstance(turns, list) and len(turns) >= 1
    for t in turns:
        assert t['speaker'] in ('caller', 'user')
        assert isinstance(t['text'], str) and t['text'].strip()
    return turns

def generate_one(task):
    """Generate one conversation according to task specification."""
    prompt = build_prompt(
        category=task['category'],
        subtype=task['desc'],
        victim_style=task.get('victim_desc'),
        lang_style=task['lang_desc'],
        min_t=task['min_t'],
        max_t=task['max_t'],
        extra=task.get('extra', ''),
    )

    for attempt in range(CONFIG['max_retries']):
        try:
            raw   = call_gpt(prompt)
            turns = parse_turns(raw)
            return {'label': task['label'], 'turns': turns}
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(2 ** attempt)
    return None

# Verify API connectivity
try:
    test_response = call_gpt('Return JSON: {"status": "ok"}', max_tokens=50)
    print(f'API connection successful')
    print(f'Test response: {test_response[:80]}')
except Exception as e:
    print(f'API connection failed: {e}')

API connection successful
Test response: {"turns":[{"caller":"เฮ้ย! สวัสดีครับ แกชนะรางวัลใหญ่จากเรานะ รับไปเลย iPhone 15


In [6]:
# Cell 6: Build task list with variation matrix
import itertools, random

random.seed(42)

# Variation matrix: cartesian product of victim x language styles.
# Cycling through this matrix ensures balanced coverage across dimensions.
victim_keys = list(VICTIM_STYLES.keys())
lang_keys   = list(LANGUAGE_STYLES.keys())
matrix      = list(itertools.product(victim_keys, lang_keys))  # 5 x 3 = 15 combinations

tasks = []

def add_scam_short_tasks():
    """SMS-style scam: 1-3 turns. Cycles through victim/lang matrix."""
    for st in SCAM_SHORT_SUBTYPES:
        for i in range(CONFIG['per_scam_short']):
            v_key, l_key = matrix[i % len(matrix)]
            tasks.append({
                'label': 1,
                'category':    f'Scam (short SMS-style) - {st}',
                'desc':        SCAM_SHORT_PATTERNS[st],
                'victim_desc': VICTIM_STYLES[v_key],
                'lang_desc':   LANGUAGE_STYLES[l_key],
                'min_t': 1, 'max_t': 3,
            })

def add_scam_long_tasks():
    """Multi-turn scam: 3-6 turns. Cycles through matrix."""
    for st in SCAM_LONG_SUBTYPES:
        for i in range(CONFIG['per_scam_long']):
            v_key, l_key = matrix[i % len(matrix)]
            tasks.append({
                'label': 1,
                'category':    f'Scam (multi-turn) - {st}',
                'desc':        SCAM_LONG_PATTERNS[st],
                'victim_desc': VICTIM_STYLES[v_key],
                'lang_desc':   LANGUAGE_STYLES[l_key],
                'min_t': 3, 'max_t': 6,
            })

def add_general_chat_tasks():
    """Casual chat with friends/family/colleagues."""
    for st in GENERAL_CHAT_SUBTYPES:
        for i in range(CONFIG['per_general_chat']):
            _, l_key = matrix[i % len(matrix)]
            tasks.append({
                'label': 0,
                'category':  'General chat',
                'desc':      GENERAL_CHAT_PATTERNS[st],
                'lang_desc': LANGUAGE_STYLES[l_key],
                'min_t': 2, 'max_t': 5,
                'extra': 'Use informal pronouns where appropriate.',
            })

def add_verification_tasks():
    """Single-turn OTP/verification messages (caller only)."""
    for st in VERIFICATION_SUBTYPES:
        for _ in range(CONFIG['per_verification']):
            tasks.append({
                'label': 0,
                'category':  'Verification code',
                'desc':      VERIFICATION_PATTERNS[st],
                'lang_desc': 'Automated formal style',
                'min_t': 1, 'max_t': 1,
                'extra': 'Caller-only message. No user response.',
            })

def add_official_tasks():
    """Calls from real institutions (banks, hospitals, government)."""
    for st in OFFICIAL_SUBTYPES:
        for i in range(CONFIG['per_official']):
            _, l_key = matrix[i % len(matrix)]
            tasks.append({
                'label': 0,
                'category':  'Official call',
                'desc':      OFFICIAL_PATTERNS[st],
                'lang_desc': LANGUAGE_STYLES[l_key],
                'min_t': 1, 'max_t': 4,
                'extra': 'Must not request OTP, payment, or sensitive data.',
            })

def add_hard_negative_tasks():
    """Legitimate calls that superficially resemble scams."""
    for st in HARD_NEGATIVE_SUBTYPES:
        for i in range(CONFIG['per_hard_negative']):
            _, l_key = matrix[i % len(matrix)]
            extra = ('Use informal pronouns. Discuss unrelated topics first.'
                     if st == 'friend_borrow' else '')
            tasks.append({
                'label': 0,
                'category':  f'Hard negative - {st}',
                'desc':      HARD_NEGATIVE_PATTERNS[st],
                'lang_desc': LANGUAGE_STYLES[l_key],
                'min_t': 2, 'max_t': 5,
                'extra': extra,
            })

def add_unknown_tasks():
    """Calls with no usable content (silent, wrong number, technical issues)."""
    for st in UNKNOWN_SUBTYPES:
        for _ in range(CONFIG['per_unknown']):
            tasks.append({
                'label': 0,
                'category':  f'Unknown - {st}',
                'desc':      UNKNOWN_PATTERNS[st],
                'lang_desc': 'Variable - depends on situation',
                'min_t': 1, 'max_t': 3,
                'extra': 'Content should be brief, unclear, or interrupted. '
                         'Not a real conversation.',
            })

add_scam_short_tasks()
add_scam_long_tasks()
add_general_chat_tasks()
add_verification_tasks()
add_official_tasks()
add_hard_negative_tasks()
add_unknown_tasks()

random.shuffle(tasks)

from collections import Counter
label_dist = Counter(t['label'] for t in tasks)
print(f'Total tasks: {len(tasks)}')
print(f'  label=1 (scam)    : {label_dist[1]} ({label_dist[1]/len(tasks)*100:.1f}%)')
print(f'  label=0 (not-scam): {label_dist[0]} ({label_dist[0]/len(tasks)*100:.1f}%)')

Total tasks: 504
  label=1 (scam)    : 246 (48.8%)
  label=0 (not-scam): 258 (51.2%)


In [7]:
# Cell 7: Generate dataset
from tqdm.notebook import tqdm

os.makedirs(CONFIG['output_dir'], exist_ok=True)
DATASET_PATH = os.path.join(CONFIG['output_dir'], CONFIG['output_file'])

successful = 0
failed = 0

with open(DATASET_PATH, 'w', encoding='utf-8') as f:
    for task in tqdm(tasks, desc='Generating'):
        record = generate_one(task)
        if record is None:
            failed += 1
            continue
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        f.flush()
        successful += 1
        time.sleep(CONFIG['request_delay_sec'])

print()
print(f'Generation complete')
print(f'  Successful: {successful}')
print(f'  Failed:     {failed}')
print(f'  Output:     {DATASET_PATH}')

Generating:   0%|          | 0/504 [00:00<?, ?it/s]


Generation complete
  Successful: 504
  Failed:     0
  Output:     /content/dataset_output/dataset.jsonl


In [8]:
# Cell 8: Statistics and quality summary
records = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

n_total = len(records)
n_scam  = sum(1 for r in records if r['label'] == 1)
n_legit = n_total - n_scam

turn_counts  = [len(r['turns']) for r in records]
text_lengths = [sum(len(t['text']) for t in r['turns']) for r in records]

len_scam  = [text_lengths[i] for i, r in enumerate(records) if r['label'] == 1]
len_legit = [text_lengths[i] for i, r in enumerate(records) if r['label'] == 0]

print('Dataset statistics')
print(f'  Total records       : {n_total}')
print(f'  Label distribution  : scam={n_scam} ({n_scam/n_total*100:.1f}%), '
      f'not-scam={n_legit} ({n_legit/n_total*100:.1f}%)')
print(f'  Turn count          : min={min(turn_counts)}, max={max(turn_counts)}, '
      f'mean={sum(turn_counts)/n_total:.2f}')
print(f'  Text length (chars) : min={min(text_lengths)}, max={max(text_lengths)}, '
      f'mean={sum(text_lengths)/n_total:.1f}')
print(f'  Mean length scam    : {sum(len_scam)/len(len_scam):.1f}')
print(f'  Mean length not-scam: {sum(len_legit)/len(len_legit):.1f}')

length_diff = abs(sum(len_scam)/len(len_scam) - sum(len_legit)/len(len_legit))
length_ratio = length_diff / (sum(len_legit)/len(len_legit)) * 100
if length_ratio > 30:
    print(f'  Warning: Length difference {length_ratio:.0f}% may cause length-based leakage')

Dataset statistics
  Total records       : 504
  Label distribution  : scam=246 (48.8%), not-scam=258 (51.2%)
  Turn count          : min=1, max=8, mean=3.78
  Text length (chars) : min=3, max=355, mean=143.6
  Mean length scam    : 174.2
  Mean length not-scam: 114.4


In [9]:
# Cell 9: Sample preview
import random as rd
rd.seed(0)

preview_indices = rd.sample(range(len(records)), 5)

for idx in preview_indices:
    r = records[idx]
    label_text = 'SCAM' if r['label'] == 1 else 'NOT_SCAM'
    print('-' * 60)
    print(f'Index: {idx} | Label: {r["label"]} ({label_text}) | Turns: {len(r["turns"])}')
    for t in r['turns']:
        print(f'  [{t["speaker"]:6s}] {t["text"]}')
    print()

------------------------------------------------------------
Index: 432 | Label: 0 (NOT_SCAM) | Turns: 1
  [caller] รหัส OTP ของคุณคือ 456789 ครับ กรุณาอย่าเผยแพร่ให้กับผู้อื่นนะครับ

------------------------------------------------------------
Index: 197 | Label: 1 (SCAM) | Turns: 7
  [caller] เฮ้ย แก ฟังนะ กูมีเรื่องด่วนมากอ่ะ
  [user  ] เอ๊ะ ลืมไปแล้ว เพื่อนใครวะ
  [caller] กูก็เปลี่ยนเบอร์ใหม่ไง อาทิตย์ก่อน
  [user  ] อ๋อ โอเค ๆ แล้วมีอะไร?
  [caller] คือกูติดปัญหาเงิน urgent ต้องโอนทันทีนะ
  [user  ] แน่ใจเหรอ? กูไม่รู้ว่าแกเปลี่ยนเบอร์ อ่ะ
  [caller] โทรกลับเบอร์เก่ากูไม่ได้หรอก ที่นี่มันยุ่งมาก

------------------------------------------------------------
Index: 388 | Label: 1 (SCAM) | Turns: 7
  [caller] เฮ้ย สวัสดีจ้า แฟนที่รัก ของฉัน
  [user  ] อื้อ มีอะไรเหรอ
  [caller] คิดถึงมากเลยนะ รอเธอไม่ไหวแล้ว
  [user  ] อืม กำลังขับรถอยู่
  [caller] อ่า รู้ๆ แต่มีเรื่องสำคัญเรื่องเงิน ฉันติดอยู่ที่นี่
  [user  ] เงิน? ทำไมล่ะ
  [caller] ฉันต้องการช่วยหน่อย ส่งเงินมาให้หน่อยได้ไหม อ่ะ

In [10]:
# Cell 10: Download output
from google.colab import files
files.download(DATASET_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>